# CXR-LT 2023 vs 2024

This notebook compares labeled CXR-LT releases at the image, study, subject, split, view, and label levels. It is intentionally separate from the per-version notebooks so cross-release assumptions are easy to audit.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

from utils import *

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 140)
pd.set_option("display.max_rows", 140)

root_dir, data_dir = get_notebook_paths()
cxr_lt_root = data_dir / "CXR-LT" / "cxr-lt-multi-label-long-tailed-classification-on-chest-x-rays-2.0.0"
cxr_lt_2023_dir = cxr_lt_root / "cxr-lt-2023"
cxr_lt_2024_dir = cxr_lt_root / "cxr-lt-2024"
mimic_cxr_jpg_dir = data_dir / "MIMIC-CXR-JPG"

print(f"cxr_lt_2023_dir: {cxr_lt_2023_dir}")
print(f"cxr_lt_2024_dir: {cxr_lt_2024_dir}")


In [ ]:
CSV_2023 = {"train": "train.csv", "development": "development.csv", "test": "test.csv"}
CSV_2024 = {"labels": "labels.csv"}
SPLIT_ALIASES = {"val": "development"}
LABEL_ALIASES_2023_TO_2024 = {"No Finding": "Normal"}

frames_2023 = load_csv_map(cxr_lt_2023_dir, CSV_2023)
df_2023_raw = pd.concat([df.assign(split=split_name) for split_name, df in frames_2023.items()], ignore_index=True)
df_2024_raw = load_csv_map(cxr_lt_2024_dir, CSV_2024)["labels"].copy()

df_2023 = df_2023_raw.rename(columns=LABEL_ALIASES_2023_TO_2024).copy()
df_2024 = df_2024_raw.copy()

for df in [df_2023, df_2024]:
    df["study_id_norm"] = df["study_id"].map(normalize_study_id_value)
    df["canonical_split"] = df["split"].replace(SPLIT_ALIASES)

labels_2023_raw = label_columns(df_2023_raw)
labels_2024 = label_columns(df_2024)
labels_2023 = [LABEL_ALIASES_2023_TO_2024.get(label, label) for label in labels_2023_raw]
shared_labels = sorted(set(labels_2023) & set(labels_2024))
labels_only_2023 = sorted(set(labels_2023) - set(labels_2024))
labels_only_2024 = sorted(set(labels_2024) - set(labels_2023))

print(f"2023 rows: {len(df_2023):,}; labels after aliasing: {len(labels_2023)}")
print(f"2024 rows: {len(df_2024):,}; labels: {len(labels_2024)}")
print(f"Shared comparable labels: {len(shared_labels)}")
print("Label aliasing for comparison:", LABEL_ALIASES_2023_TO_2024)


## 1) Dataset Size And Identifier Overlap


In [ ]:
def release_counts(name, df):
    return {
        "release": name,
        "rows": len(df),
        "subjects": df["subject_id"].nunique(),
        "studies": df["study_id_norm"].nunique(),
        "dicoms": df["dicom_id"].nunique(),
    }

counts_df = pd.DataFrame([release_counts("2023", df_2023), release_counts("2024", df_2024)])
display(counts_df)

identifier_overlap_df = pd.DataFrame(
    [
        {
            "identifier": key,
            "2023_unique": df_2023[col].nunique(),
            "2024_unique": df_2024[col].nunique(),
            "overlap": len(set(df_2023[col]) & set(df_2024[col])),
            "only_2023": len(set(df_2023[col]) - set(df_2024[col])),
            "only_2024": len(set(df_2024[col]) - set(df_2023[col])),
        }
        for key, col in [("subject_id", "subject_id"), ("study_id_norm", "study_id_norm"), ("dicom_id", "dicom_id")]
    ]
)
identifier_overlap_df["overlap_pct_of_2023"] = identifier_overlap_df["overlap"] / identifier_overlap_df["2023_unique"] * 100
identifier_overlap_df["overlap_pct_of_2024"] = identifier_overlap_df["overlap"] / identifier_overlap_df["2024_unique"] * 100
display(identifier_overlap_df)

if (identifier_overlap_df[["only_2023", "only_2024"]].sum().sum() == 0):
    print("The 2023 and 2024 labeled releases contain the same subjects, studies, and DICOM images. Differences below are label/split taxonomy differences, not image-cohort differences.")


## 2) Label Set Differences


In [ ]:
label_set_df = pd.DataFrame(
    [
        {"bucket": "shared", "count": len(shared_labels), "labels": ", ".join(shared_labels)},
        {"bucket": "only_2023", "count": len(labels_only_2023), "labels": ", ".join(labels_only_2023)},
        {"bucket": "only_2024", "count": len(labels_only_2024), "labels": ", ".join(labels_only_2024)},
    ]
)
display(label_set_df)

label_presence_df = pd.DataFrame(
    [
        {"label": label, "in_2023": label in labels_2023, "in_2024": label in labels_2024}
        for label in sorted(set(labels_2023) | set(labels_2024))
    ]
)
display(label_presence_df)


## 3) Split Composition Differences


In [ ]:
split_counts_df = pd.concat(
    [
        df_2023.groupby(["split", "canonical_split"]).agg(rows=("dicom_id", "size"), subjects=("subject_id", "nunique"), studies=("study_id_norm", "nunique"), dicoms=("dicom_id", "nunique")).assign(release="2023").reset_index(),
        df_2024.groupby(["split", "canonical_split"]).agg(rows=("dicom_id", "size"), subjects=("subject_id", "nunique"), studies=("study_id_norm", "nunique"), dicoms=("dicom_id", "nunique")).assign(release="2024").reset_index(),
    ],
    ignore_index=True,
)
split_counts_df["row_pct_within_release"] = split_counts_df["rows"] / split_counts_df.groupby("release")["rows"].transform("sum") * 100
display(split_counts_df)

plt.figure(figsize=(8, 4.5))
sns.barplot(data=split_counts_df, x="canonical_split", y="row_pct_within_release", hue="release")
plt.title("Canonical split row share by release")
plt.xlabel("")
plt.ylabel("Rows within release (%)")
plt.tight_layout()
plt.show()

split_assignment_df = df_2023[["dicom_id", "canonical_split"]].merge(
    df_2024[["dicom_id", "canonical_split"]],
    on="dicom_id",
    suffixes=("_2023", "_2024"),
    validate="one_to_one",
)
split_transition_df = pd.crosstab(
    split_assignment_df["canonical_split_2023"],
    split_assignment_df["canonical_split_2024"],
    margins=True,
)
display(split_transition_df)

changed_split_df = split_assignment_df.query("canonical_split_2023 != canonical_split_2024")
print(f"Images with changed canonical split assignment: {len(changed_split_df):,} / {len(split_assignment_df):,}")
display(changed_split_df.head(20))


In [ ]:
split_order = ["train", "development", "test"]
split_labels = ["train", "dev", "test"]

split_overlap_matrix_df = pd.crosstab(
    split_assignment_df["canonical_split_2023"],
    split_assignment_df["canonical_split_2024"],
).reindex(index=split_order, columns=split_order, fill_value=0)

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(split_overlap_matrix_df.to_numpy(), cmap="Blues")
fig.colorbar(im, ax=ax, label="DICOM overlap count")

ax.set_xticks(np.arange(len(split_labels)))
ax.set_yticks(np.arange(len(split_labels)))
ax.set_xticklabels(split_labels)
ax.set_yticklabels(split_labels)
ax.set_xlabel("2024 split")
ax.set_ylabel("2023 split")
ax.set_title("CXR-LT 2023 vs 2024 Split Overlap")

threshold = split_overlap_matrix_df.to_numpy().max() / 2
for row_idx in range(len(split_labels)):
    for col_idx in range(len(split_labels)):
        count = split_overlap_matrix_df.iat[row_idx, col_idx]
        color = "white" if count > threshold else "black"
        ax.text(col_idx, row_idx, f"{count:,}", ha="center", va="center", color=color)

plt.grid(False)
plt.tight_layout()
plt.show()


## 4) View-Position Differences


In [ ]:
view_2023 = summarize_view_positions(df_2023, "2023", top_n=999).rename(columns={"count": "count_2023", "rate_pct": "rate_pct_2023"}).drop(columns=["split"])
view_2024 = summarize_view_positions(df_2024, "2024", top_n=999).rename(columns={"count": "count_2024", "rate_pct": "rate_pct_2024"}).drop(columns=["split"])
view_diff_df = view_2023.merge(view_2024, on="ViewPosition", how="outer").fillna(0)
view_diff_df["count_delta_2024_minus_2023"] = view_diff_df["count_2024"] - view_diff_df["count_2023"]
view_diff_df["rate_delta_pct_points_2024_minus_2023"] = view_diff_df["rate_pct_2024"] - view_diff_df["rate_pct_2023"]
view_diff_df["abs_rate_delta_pct_points"] = view_diff_df["rate_delta_pct_points_2024_minus_2023"].abs()
view_diff_df = view_diff_df.sort_values(["abs_rate_delta_pct_points", "ViewPosition"], ascending=[False, True])
display(view_diff_df)

if (view_diff_df["count_delta_2024_minus_2023"].abs().sum() == 0):
    print("View-position distributions are identical because both releases contain the same image rows. Skipping the zero-delta plot.")
else:
    plot_df = view_diff_df.query("abs_rate_delta_pct_points > 0").head(15).sort_values("rate_delta_pct_points_2024_minus_2023")
    plt.figure(figsize=(9, max(4, len(plot_df) * 0.35)))
    sns.barplot(data=plot_df, x="rate_delta_pct_points_2024_minus_2023", y="ViewPosition", color="C1")
    plt.axvline(0, color="black", linewidth=1)
    plt.title("Largest view-position shifts: 2024 minus 2023")
    plt.xlabel("Percentage-point difference")
    plt.ylabel("")
    plt.tight_layout()
    plt.show()


## 6) Image Path Availability


In [ ]:
image_root = mimic_cxr_jpg_dir
path_summary_df = pd.concat(
    [
        summarize_image_paths({"2023": df_2023.drop(columns=["study_id_norm"])}, image_root),
        summarize_image_paths({"2024": df_2024.drop(columns=["study_id_norm"])}, image_root),
    ],
    ignore_index=True,
).rename(columns={"split": "release"})
display(path_summary_df)

path_col_2023 = detect_path_column(df_2023)
path_col_2024 = detect_path_column(df_2024)
path_overlap_df = pd.DataFrame(
    [
        {
            "identifier": "image_path",
            "2023_unique": df_2023[path_col_2023].nunique(),
            "2024_unique": df_2024[path_col_2024].nunique(),
            "overlap": len(set(df_2023[path_col_2023].dropna()) & set(df_2024[path_col_2024].dropna())),
            "only_2023": len(set(df_2023[path_col_2023].dropna()) - set(df_2024[path_col_2024].dropna())),
            "only_2024": len(set(df_2024[path_col_2024].dropna()) - set(df_2023[path_col_2023].dropna())),
        }
    ]
)
display(path_overlap_df)


## Readout

Use the identifier overlap to decide whether 2024 can replace 2023 for experiments, and use the shared-label prevalence deltas to choose whether model comparisons should be restricted to labels common to both releases.
